In [3]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
con = duckdb.connect(str(PROJECT_ROOT / "data" / "f1.db"), read_only=True)

# Confirm connection
print(con.execute("SELECT COUNT(*) FROM f1_results").fetchone()[0], "results rows ready")

25939 results rows ready


In [11]:
## Notebook 2: Text Generation

#RAG systems embed and retrieve text, not rows.
#This notebook converts structured F1 data into natural language sentences.

#The quality of these sentences directly determines retrieval quality.
#Better sentences = better RAG answers.

#We generate one sentence type per table, then combine into a single corpus file.

In [5]:
def result_to_text(row):
    position = row['position']
    grid = row['grid']
    
    # Handle DNFs
    try:
        pos_int = int(float(position))
        if pos_int == 1:
            pos_str = "1st (winner)"
        elif pos_int == 2:
            pos_str = "2nd"
        elif pos_int == 3:
            pos_str = "3rd"
        else:
            pos_str = f"{pos_int}th"
    except:
        pos_str = f"retired ({row['status']})"

    # Handle grid position
    try:
        grid_int = int(float(grid))
        grid_str = f"starting from position {grid_int}"
    except:
        grid_str = "from an unknown grid position"

    # Fastest lap note
    fastest = ""
    try:
        if int(float(row['fastest_lap_rank'])) == 1:
            fastest = " and set the fastest lap"
    except:
        pass

    return (
        f"In the {row['season']} {row['race_name']}, "
        f"{row['driver_name']} finished {pos_str} {grid_str}, "
        f"scoring {row['points']} points for {row['constructor']}{fastest}."
    )

# Test on 5 rows
sample = con.execute("SELECT * FROM f1_results LIMIT 5").df()
for _, row in sample.iterrows():
    print(result_to_text(row))
    print()

In the 1950 British Grand Prix, Nino Farina finished 1st (winner) starting from position 1, scoring 9.0 points for Alfa Romeo.

In the 1950 British Grand Prix, Luigi Fagioli finished 2nd starting from position 2, scoring 6.0 points for Alfa Romeo.

In the 1950 British Grand Prix, Reg Parnell finished 3rd starting from position 4, scoring 4.0 points for Alfa Romeo.

In the 1950 British Grand Prix, Yves Cabantous finished 4th starting from position 6, scoring 3.0 points for Talbot-Lago.

In the 1950 British Grand Prix, Louis Rosier finished 5th starting from position 9, scoring 2.0 points for Talbot-Lago.



In [6]:
results_df = con.execute("SELECT * FROM f1_results").df()
result_sentences = [result_to_text(row) for _, row in results_df.iterrows()]

print(f"Generated {len(result_sentences)} result sentences")
print("\nSample:")
print(result_sentences[0])
print(result_sentences[100])
print(result_sentences[1000])

Generated 25939 result sentences

Sample:
In the 1950 British Grand Prix, Nino Farina finished 1st (winner) starting from position 1, scoring 9.0 points for Alfa Romeo.
In the 1950 Belgian Grand Prix, Nino Farina finished 4th starting from position 1, scoring 4.0 points for Alfa Romeo.
In the 1954 Spanish Grand Prix, Paco Godia finished 6th starting from position 13, scoring 0.0 points for Maserati.


In [7]:
def standing_to_text(row):
    # Skip rows with null position
    if pd.isna(row['position']):
        return None
    
    pos = int(float(row['position']))
    if pos == 1:
        pos_str = "led"
    else:
        suffix = 'st' if pos==1 else 'nd' if pos==2 else 'rd' if pos==3 else 'th'
        pos_str = f"was {pos}{suffix} in"

    return (
        f"At the end of the {row['season']} season, "
        f"{row['driver_id']} {pos_str} the drivers championship "
        f"with {row['points']} points and {row['wins']} wins, "
        f"driving for {row['constructor']}."
    )

standings_df = con.execute("SELECT * FROM f1_driver_standings").df()

# Generate and filter out None values
standing_sentences = [
    s for s in (standing_to_text(row) for _, row in standings_df.iterrows())
    if s is not None
]

print(f"Generated {len(standing_sentences)} standing sentences")
print("\nSample:")
for s in standing_sentences[:3]:
    print(s)
    print()

Generated 1664 standing sentences

Sample:
At the end of the 1950 season, farina led the drivers championship with 30.0 points and 3 wins, driving for Alfa Romeo.

At the end of the 1950 season, fangio was 2nd in the drivers championship with 27.0 points and 3 wins, driving for Alfa Romeo.

At the end of the 1950 season, fagioli was 3rd in the drivers championship with 24.0 points and 0 wins, driving for Alfa Romeo.



In [8]:
def constructor_standing_to_text(row):
    if pd.isna(row['position']):
        return None
    
    pos = int(float(row['position']))
    suffix = 'st' if pos==1 else 'nd' if pos==2 else 'rd' if pos==3 else 'th'
    return (
        f"In the {row['season']} constructors championship, "
        f"{row['constructor']} finished {pos}{suffix} "
        f"with {row['points']} points and {row['wins']} wins."
    )

constructor_standings_df = con.execute("SELECT * FROM f1_constructor_standings").df()

constructor_sentences = [
    s for s in (constructor_standing_to_text(row) for _, row in constructor_standings_df.iterrows())
    if s is not None
]

print(f"Generated {len(constructor_sentences)} constructor standing sentences")
print("\nSample:")
for s in constructor_sentences[:3]:
    print(s)
    print()

Generated 712 constructor standing sentences

Sample:
In the 1958 constructors championship, Vanwall finished 1st with 48.0 points and 6 wins.

In the 1958 constructors championship, Ferrari finished 2nd with 40.0 points and 2 wins.

In the 1958 constructors championship, Cooper finished 3rd with 31.0 points and 2 wins.



In [9]:
all_sentences = result_sentences + standing_sentences + constructor_sentences

print(f"Total corpus size: {len(all_sentences)} sentences")
print(f"  - Result sentences:               {len(result_sentences)}")
print(f"  - Driver standing sentences:      {len(standing_sentences)}")
print(f"  - Constructor standing sentences: {len(constructor_sentences)}")

# Save to processed/
output_path = PROJECT_ROOT / "data" / "processed" / "corpus.txt"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    for sentence in all_sentences:
        f.write(sentence + "\n")

print(f"\nCorpus saved to {output_path}")

Total corpus size: 28315 sentences
  - Result sentences:               25939
  - Driver standing sentences:      1664
  - Constructor standing sentences: 712

Corpus saved to /home/humza/projects/f1-rag/data/processed/corpus.txt


In [10]:
# Spot check for empty or broken sentences
empty = [s for s in all_sentences if len(s) < 20]
print(f"Suspiciously short sentences: {len(empty)}")
if empty:
    for s in empty[:5]:
        print(repr(s))

# Length distribution
lengths = [len(s) for s in all_sentences]
print(f"\nSentence length stats:")
print(f"  Min:    {min(lengths)} chars")
print(f"  Max:    {max(lengths)} chars")
print(f"  Avg:    {sum(lengths)//len(lengths)} chars")

Suspiciously short sentences: 0

Sentence length stats:
  Min:    83 chars
  Max:    161 chars
  Avg:    120 chars
